# 🧬 Genome-Doc — SIR Training (Colab)

This notebook trains the **Style & Identity Refiner (SIR)** module using contrastive learning.

**Steps:**
1. Upload project & install dependencies
2. Upload or generate synthetic dataset
3. Smoke test (--test mode)
4. Full training

**Runtime:** Set to **GPU → T4** (Runtime → Change runtime type)

## 1️⃣ Check GPU

In [ ]:
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 2️⃣ Upload Project

**The Two-ZIP Extraction Method:**
To save time uploading the massive synthetic dataset repeatedly, we use two separate zips stored in your Google Drive:
1. genome-doc.zip - Contains the full project including the 30K dataset.
2. genome-cleaned.zip - Contains ONLY the updated Python code and configs.

Run the cells below in order.

In [ ]:
# === STEP 1: Extract BASE Project + Dataset ===
from google.colab import drive
drive.mount('/content/drive')

# Removing the -q (quiet) flag so you can see all extraction logs
!unzip -o /content/drive/MyDrive/PROMETHEUS/genome-doc.zip -d /content/

!echo "\n✅ Extraction of Dataset Base Complete!\n"
!ls /content/genome-doc/

In [ ]:
# === STEP 2: Extract UPDATED Code ===
# This overwrites the older code files with the light-weight cleaned zip.

!unzip -o /content/drive/MyDrive/PROMETHEUS/genome-cleaned.zip -d /content/

!echo "\n✅ Codebase updated with latest files!\n"
!ls /content/genome-doc/

In [ ]:
# Set working directory
import os
os.chdir('/content/genome-doc')
!pwd
!ls

## 3️⃣ Install Dependencies
We use verbose flags (-v or dropping -q) to track exactly what pip is installing.

In [ ]:
!pip install pyyaml pillow numpy tqdm pydantic
!pip install torch torchvision --upgrade
!pip install lpips  # For perceptual loss (optional)
print("\n✅ Dependencies installed! Review the logs above.")

## 4️⃣ Generate Small Test Dataset (if not uploaded)

Skip this if you already uploaded your 30K dataset inside the zip.

In [ ]:
# Check if dataset already exists
import os
dataset_exists = os.path.exists('data/synthetic/train/images/clean')
print(f"Checking for dataset... Exists: {dataset_exists}")

if dataset_exists:
    num_clean = len(os.listdir('data/synthetic/train/images/clean'))
    has_degraded = os.path.exists('data/synthetic/train/images/degraded')
    num_degraded = len(os.listdir('data/synthetic/train/images/degraded')) if has_degraded else 0
    num_genomes = len(os.listdir('data/synthetic/train/genomes'))
    print(f"  -> Found Clean images: {num_clean}")
    print(f"  -> Found Degraded images: {num_degraded}")
    print(f"  -> Found Genomes: {num_genomes}")

In [ ]:
# Generate a small dataset for testing (200 samples, ~1 min)
# SKIP this cell if you already have your dataset!

!python data/synthetic_generator.py --num-samples 200 --output-dir data/synthetic/train
print("\n✅ Clean dataset generated!")

## 5️⃣ Smoke Test (--test mode)

Runs **2 epochs on 100 samples** to verify the full pipeline works.  
Should complete in **~1-2 minutes** on T4.

In [ ]:
!python training/train_sir.py \
    --config configs/sir_resnet.yaml \
    --data-dir data/synthetic/train \
    --output-dir checkpoints/sir \
    --test

## 6️⃣ Full Training (Optional)

Only run this if:
- ✅ Smoke test passed
- ✅ You have the full 30K dataset uploaded
- ✅ You have enough Colab GPU time

**Estimated time on T4 (batch=16):** ~1-2 days (100 epochs)  
**Estimated time on RTX 6000 Ada / Blackwell 96GB (batch=256):** ~4-6 hours

In [ ]:
# Full SIR training
!python training/train_sir.py \
    --config configs/sir_resnet.yaml \
    --data-dir data/synthetic/train \
    --output-dir checkpoints/sir

## 7️⃣ Download Checkpoint

In [ ]:
# Check what checkpoints were saved
!ls -lh checkpoints/sir/

In [ ]:
# Download best checkpoint to local machine
from google.colab import files
files.download('checkpoints/sir/best_model.pt')

In [ ]:
# Or save to Google Drive
!cp -r checkpoints/sir/ /content/drive/MyDrive/genome-doc-checkpoints/sir/

print("✅ Checkpoints saved to Google Drive!")